# Asynchronous Sigma-Delta Modulation

Simulation results for the asynchronous sigma-delta modulator chapter.

For an integrator, a first-order low-pass and a second-order low-pass loop filter, the
DC characteristic of the modulator is simulated and characterized by the Taylor
coefficients of the average output and of the oscillation frequency. The simulated
coefficients are compared with the predictions of the nonlinear model.

In [ ]:
import os

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from delta_sigma_simulator.modulator import DeltaSigmaModulator
from delta_sigma_simulator.filter import (
    FilterIntegrator,
    FilterFirstOrder,
    FilterSecondOrder,
    FilterButterworth,
)
from delta_sigma_simulator.quantizer import QuantizerDelayHysteresis
from delta_sigma_simulator.wave import BinaryWave, SineWave

In [ ]:
# - Directory the generated CSV files are written to. Set the BOOK_TIKZ environment
#   variable to the assets/tikz directory of the book to update its figures directly.
out = Path(os.environ.get("BOOK_TIKZ", "."))
# - Self-oscillation frequency
f0 = 1.0
# - Ratio of the corner frequency of the loop filter to the self-oscillation frequency,
#   for the simulated and for the predicted values
beta0_sim = 1 / np.logspace(0, 3, 20)
beta0 = 1 / np.logspace(0, 3, 1000)
# - The general loop filter approximation requires a corner frequency well below the
#   self-oscillation frequency, so the second-order low-pass filter is only evaluated on
#   the lower half of that range
beta0_gen_sim = 1 / np.logspace(np.log10(2), 3, 20)
beta0_gen = 1 / np.logspace(np.log10(2), 3, 1000)
# - Amplitude of the average output used to extract the Taylor coefficients, and the
#   number of points it is sampled with. A larger amplitude improves the conditioning of
#   the fit, a smaller one reduces the influence of the higher-order terms.
V_max = 0.1
n_V = 21

## DC characteristic

For a static input, the modulator settles into a limit cycle. The oscillation
frequency `f` follows from the period of the limit cycle and the average output `V`
from its duty cycle. Sweeping the static input yields the DC characteristic of the
modulator, which is characterized by the Taylor coefficients `V_n` of the average
output and by the Taylor coefficients `f_n` of the oscillation frequency, both taken
with respect to the static input.

In [ ]:
def limit_cycle(asdm, U, n):
    """
    Simulate the modulator for a static input and return the oscillation frequency and
    the average output of the resulting limit cycle.

    Arguments:
        asdm: The modulator to simulate.
        U: The static input.
        n: The number of transitions to simulate.
    """
    e = asdm.simulate([BinaryWave(E=U)], n=n).e

    assert np.isclose(e[-1] - e[-3], e[-3] - e[-5]), "Simulation failed to converge."

    # - The output is +1 on the interval (e[-3], e[-2]) when it is preceded by an even
    #   number of transitions, and -1 otherwise
    s = (-1.0) ** (len(e) - 3)

    f = 1 / (e[-1] - e[-3])
    V = s * (2 * (e[-2] - e[-3]) / (e[-1] - e[-3]) - 1)

    return f, V


def dc_characteristic(asdm, U, n):
    """
    Simulate the DC characteristic of the modulator.

    Arguments:
        asdm: The modulator to simulate.
        U: Array of static inputs.
        n: The number of transitions to simulate per static input.
    """
    f = np.zeros_like(U)
    V = np.zeros_like(U)

    for i, U_i in enumerate(U):
        f[i], V[i] = limit_cycle(asdm, U_i, n)

    return f, V


def taylor(x, y, n=7):
    """
    Return the first n + 1 Taylor coefficients of y as a function of x, obtained by
    fitting a polynomial of degree n. The abscissa is normalized before fitting to keep
    the system well conditioned.

    Arguments:
        x: Array of abscissa values, centred around zero.
        y: Array of ordinate values.
        n: Degree of the polynomial.
    """
    x0 = np.max(np.abs(x))

    c = np.polynomial.polynomial.polyfit(x / x0, y, n)

    return c / x0 ** np.arange(n + 1)


def coefficients(U, V, f):
    """
    Taylor coefficients f_0, f_2, V_1 and V_3 of a simulated DC characteristic, both
    taken with respect to the static input. The coefficients of the average output are
    obtained by inverting the series of the static input.

    Arguments:
        U: Array of static inputs.
        V: Array of average outputs.
        f: Array of oscillation frequencies.
    """
    c_f = taylor(U, f)
    c_U = taylor(V, U)

    return c_f[0], c_f[2], 1 / c_U[1], -c_U[3] / c_U[1] ** 4


def plot_amplitude(beta0, c, beta0_sim, c_sim):
    """
    Plot the predicted and the simulated value of V_1 and V_3.
    """
    plt.loglog(beta0, np.abs(c[3]), "k-", label=r"$V_1$")
    plt.loglog(beta0_sim, np.abs(c_sim[3]), "ko", label=r"$V_{1,\mathrm{sim}}$")
    plt.loglog(beta0, np.abs(c[4]), "-", color="gray", label=r"$V_3$")
    plt.loglog(
        beta0_sim, np.abs(c_sim[4]), "o", color="gray", label=r"$V_{3,\mathrm{sim}}$"
    )
    plt.xlabel(r"$\beta_0$")
    plt.ylabel(r"$\left|.\right|$")
    plt.grid(which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.show()


def plot_frequency(beta0, c, beta0_sim, c_sim):
    """
    Plot the predicted and the simulated value of f_0 and f_2.
    """
    plt.semilogx(beta0, c[1], "k-", label=r"$f_0$")
    plt.semilogx(beta0_sim, c_sim[1], "ko", label=r"$f_{0,\mathrm{sim}}$")
    plt.semilogx(beta0, c[2] / c[1], "-", color="gray", label=r"$f_2/f_0$")
    plt.semilogx(
        beta0_sim,
        c_sim[2] / c_sim[1],
        "o",
        color="gray",
        label=r"$f_{2,\mathrm{sim}}/f_{0,\mathrm{sim}}$",
    )
    plt.xlabel(r"$\beta_0$")
    plt.ylim(-2, +2)
    plt.grid(which="both", ls="--", alpha=0.5)
    plt.legend()
    plt.show()

## General loop filter approximation

For a loop filter that is approximately first order at high frequencies, the nonlinear
model gives the average output through the Taylor coefficients

    V_1 = 1 / (1 - s x)     and     V_3 = s x V_1^4,

where `x = pi^2 alpha0 beta0 / 6`, and the oscillation frequency in closed form as
`f = f0 (1 - V^2)`, so that `f_2 = -f0 V_1^2` with respect to the static input. The
sign `s` is that of the real part of the loop filter transfer function at the
self-oscillation frequency, which selects the upper or the lower sign of the design
equations, since the real part is what the even part of the loop filter response
follows. The hysteresis follows from `A_0 = 2 delta / pi`, independently of `s`.

In [ ]:
def general_coefficients(beta0, alpha0, s):
    """
    Taylor coefficients f_0, f_2, V_1 and V_3 predicted by the general loop filter
    approximation, both taken with respect to the static input.

    Arguments:
        beta0: Ratio of the corner frequency to the self-oscillation frequency.
        alpha0: Ratio of the gain at the self-oscillation frequency to the DC gain.
        s: Sign of the real part of the loop filter transfer function.
    """
    x = np.pi**2 / 6 * alpha0 * beta0

    V_1 = 1 / (1 - s * x)
    V_3 = s * x * V_1**4

    return f0 * np.ones_like(V_1), -f0 * V_1**2, V_1, V_3


def general_hysteresis(A_0):
    """
    Hysteresis that yields the desired self-oscillation frequency.

    Arguments:
        A_0: Gain of the loop filter at the self-oscillation frequency.
    """
    return np.pi / 2 * A_0

## Integrator loop filter

The loop filter is an integrator with unity gain at the self-oscillation frequency.
The hysteresis follows from the design equation of the nonlinear model.
The integrator has no corner frequency, so there is nothing to sweep against:
the DC characteristic reduces to `V = U` and the oscillation frequency to
`f = f0 (1 - V^2)`, giving `V_1 = 1`, `V_3 = 0`, `f_0` and `f_2 = -f0` exactly. A
single simulation therefore suffices, and its deviation from those values measures
the numerical accuracy of the simulator.


In [ ]:
# - Gain of the loop filter at the self-oscillation frequency
A_0 = 1.0
# - Hysteresis that yields the desired self-oscillation frequency
d = general_hysteresis(A_0)

quantizer = QuantizerDelayHysteresis(0.0, d)
quantizer.t_step = 0.1 / f0

asdm = DeltaSigmaModulator(FilterIntegrator(2 * np.pi * f0 * A_0), quantizer)

U = V_max * np.linspace(-1, 1, n_V)

f, V = dc_characteristic(asdm, U, n=20)

# - Exact coefficients, against which the simulated ones are checked
c_int = np.array([f0, -f0, 1.0, 0.0])
c_int_sim = np.array(coefficients(U, V, f))

for name, sim, exact in zip(("f_0", "f_2", "V_1", "V_3"), c_int_sim, c_int):
    print(f"{name}: simulated {sim:+.15f}, exact {exact:+.1f}, error {abs(sim - exact):.2e}")

print(f"worst error: {np.max(np.abs(c_int_sim - c_int)):.2e}")


## First-order low-pass loop filter

The loop filter is a first-order low-pass filter with unity DC gain and a corner
frequency `beta0 f0`. Besides the exact prediction, the prediction of the general loop
filter approximation is evaluated as well. The gain of that approximation at the
self-oscillation frequency is `beta0` times the DC gain, and the real part of the
transfer function at the self-oscillation frequency is positive.

In [ ]:
# - DC gain of the loop filter
A_DC = 1.0

# - Exact prediction
w = np.pi * beta0 / 2 / np.tanh(np.pi * beta0 / 2)

V_1 = np.sinh(np.pi * beta0) / (np.pi * beta0)
V_3 = (np.pi**2 * beta0**2 / 12 + w * (w - 1)) * V_1**3

# - Prediction of the general loop filter approximation
_, _, V_1_gen, V_3_gen = general_coefficients(beta0, beta0, +1.0)

c_lpf = np.stack(
    [1 / beta0, f0 * np.ones_like(beta0), -f0 * w * V_1**2, V_1, V_3, V_1_gen, V_3_gen]
)

c_lpf_sim = np.zeros((5, len(beta0_sim)))

for i, beta0_i in enumerate(beta0_sim):
    # - Hysteresis that yields the desired self-oscillation frequency
    d = A_DC * np.tanh(np.pi * beta0_i / 2)

    quantizer = QuantizerDelayHysteresis(0.0, d)
    quantizer.t_step = 0.1 / f0

    asdm = DeltaSigmaModulator(FilterFirstOrder(beta0_i * f0, A_DC), quantizer)

    U = V_max / (np.sinh(np.pi * beta0_i) / (np.pi * beta0_i)) * np.linspace(-1, 1, n_V)

    f, V = dc_characteristic(asdm, U, n=20)

    c_lpf_sim[:, i] = 1 / beta0_i, *coefficients(U, V, f)

np.savetxt(out / "asdm-loop-filter-low-pass-expression.csv", c_lpf.T, delimiter=",")
np.savetxt(out / "asdm-loop-filter-low-pass-simulation.csv", c_lpf_sim.T, delimiter=",")

In [ ]:
plot_amplitude(beta0, c_lpf, beta0_sim, c_lpf_sim)
plot_frequency(beta0, c_lpf, beta0_sim, c_lpf_sim)

## Second-order low-pass loop filter

The loop filter is a second-order low-pass filter with unity DC gain, a zero at
`4 beta0 f0` and poles at `2 beta0 f0` and `beta0 f0`. Its gain at the self-oscillation
frequency is `beta0 / 2` times the DC gain, and the real part of its transfer function
at the self-oscillation frequency is negative. No exact prediction is available for
this loop filter, so the general loop filter approximation is used instead.

In [ ]:
# - DC gain of the loop filter
A_DC = 1.0
# - Sign of the real part of the transfer function at the self-oscillation frequency
s = -1.0

c_gen = np.stack([1 / beta0_gen, *general_coefficients(beta0_gen, beta0_gen / 2, s)])

c_gen_sim = np.zeros((5, len(beta0_gen_sim)))

for i, beta0_i in enumerate(beta0_gen_sim):
    # - Hysteresis that yields the desired self-oscillation frequency
    d = general_hysteresis(beta0_i / 2 * A_DC)

    quantizer = QuantizerDelayHysteresis(0.0, d)
    quantizer.t_step = 0.1 / f0

    asdm = DeltaSigmaModulator(
        FilterSecondOrder(4 * beta0_i * f0, 2 * beta0_i * f0, beta0_i * f0, A_DC),
        quantizer,
    )

    U = V_max * np.linspace(-1, 1, n_V)

    # - The limit cycle of a second-order loop filter settles over roughly half a time
    #   constant of its slowest pole, so the number of transitions is scaled accordingly
    f, V = dc_characteristic(asdm, U, n=int(max(20, 1 / beta0_i)))

    c_gen_sim[:, i] = 1 / beta0_i, *coefficients(U, V, f)

np.savetxt(out / "asdm-loop-filter-general-expression.csv", c_gen.T, delimiter=",")
np.savetxt(out / "asdm-loop-filter-general-simulation.csv", c_gen_sim.T, delimiter=",")

In [ ]:
plot_amplitude(beta0_gen, c_gen, beta0_gen_sim, c_gen_sim)
plot_frequency(beta0_gen, c_gen, beta0_gen_sim, c_gen_sim)

## Input frequency

The DC characteristic assumes a static input. Replacing it by a sinusoidal input of
frequency `f_u` and amplitude `U` and demodulating the output gives the third-order
output intercept point as a function of the input frequency: the output is low-pass
filtered by a tenth-order Butterworth filter with a corner frequency of `4 f_u`,
sampled at sixteen times that corner frequency, and the fundamental and the third
harmonic are read from its discrete Fourier transform.

The order of the demodulation filter is a compromise. It has to reject the limit cycle,
but the state of the filter is expanded over its poles, and that expansion loses
accuracy as the order grows: at the scale of this measurement a tenth-order filter
resolves the third harmonic to a relative error of `1e-4`, a sixteenth-order one only
to `0.15`.

In [ ]:
def oip3_frequency(asdm, f_u, U, n_settle=4, n_period=4, order=10):
    """
    Third-order output intercept point of the modulator for a sinusoidal input, obtained
    by demodulating the output and reading the fundamental and the third harmonic from
    its discrete Fourier transform.

    Arguments:
        asdm: The modulator to simulate.
        f_u: Frequency of the sinusoidal input.
        U: Amplitude of the sinusoidal input.
        n_settle: Input periods discarded to settle the modulator and the demodulation
            filter.
        n_period: Input periods the transform is taken over.
        order: Order of the demodulation filter.
    """
    v = asdm.simulate([SineWave([0, U], f_u)], t=(n_settle + n_period) / f_u)

    # - Corner frequency of the demodulation filter and sampling frequency
    f_c = 4 * f_u
    f_s = 16 * f_c

    h = FilterButterworth(order, f_c)

    t = n_settle / f_u + np.arange(0.0, n_period / f_u - 0.5 / f_s, 1 / f_s)

    V = np.fft.rfft(v.filter(h)(t)) / (len(t) / 2)

    # - The demodulation filter attenuates the third harmonic slightly more than the
    #   fundamental, which is compensated for
    H = h.frequency_response(2 * np.pi * np.array([f_u, 3 * f_u]))

    V_1 = np.abs(V[n_period] / H[0]) / U
    V_3 = 4 * np.abs(V[3 * n_period] / H[1]) / U**3

    return 20 * np.log10(np.sqrt(2 * np.abs(V_1**3 / V_3)))

In [ ]:
# - The quantizer of this chapter is ideal
t_d = 0.0
# - Prefix of the generated CSV files
prefix = "asdm"


def loop_filters(beta0):
    """
    The integrator and the first-order low-pass loop filter, each with the hysteresis
    that yields the desired self-oscillation frequency.

    Arguments:
        beta0: Ratio of the corner frequency to the self-oscillation frequency.
    """
    A_DC = 1.0

    return (
        (FilterIntegrator(2 * np.pi * f0 * 1.0), general_hysteresis(1.0)),
        (FilterFirstOrder(beta0 * f0, A_DC), A_DC * np.tanh(np.pi * beta0 / 2)),
    )

In [ ]:
# - Amplitude of the sinusoidal input
U_u = 0.1
# - Ratio of the corner frequency of the loop filter to the self-oscillation frequency
beta0_u = 1e-2
# - Input frequencies, expressed as the number of self-oscillation periods they span
K = np.unique(np.round(np.logspace(1, 4, 12)).astype(int))

c_fu = np.zeros((4, len(K)))

for i, K_i in enumerate(K):
    for j, (h, d) in enumerate(loop_filters(beta0_u)):
        quantizer = QuantizerDelayHysteresis(t_d, d)
        quantizer.t_step = 0.1 / f0

        asdm = DeltaSigmaModulator(h, quantizer)

        c_fu[j + 1, i] = oip3_frequency(asdm, f0 / K_i, U_u)

c_fu[0] = 1 / K

# - Intercept point of the first-order low-pass loop filter that follows from its DC
#   characteristic, which the simulation has to approach at a low input frequency
w_u = np.pi * beta0_u / 2 / np.tanh(np.pi * beta0_u / 2)

c_fu[3] = 20 * np.log10(np.sqrt(2 / (np.pi**2 * beta0_u**2 / 12 + w_u * (w_u - 1))))

np.savetxt(
    out / f"{prefix}-oip3-frequency.csv",
    c_fu.T,
    delimiter=",",
    header="fu_over_f0,OIP3_integrator,OIP3_low_pass,OIP3_low_pass_static",
    comments="",
)

In [ ]:
# - Slope of each curve, to detect the roll-off with the input frequency
for name, y in (("integrator", c_fu[1]), ("low-pass", c_fu[2])):
    slope = np.diff(y) / np.diff(np.log10(c_fu[0]))

    print(
        f"{name:>11}: {y.max():6.1f} to {y.min():6.1f} dB, slope "
        f"{np.mean(slope[:4]):+6.1f} dB/dec over the highest decade, "
        f"{np.mean(slope[-4:]):+6.1f} dB/dec over the lowest"
    )

In [ ]:
plt.semilogx(c_fu[0], c_fu[1], "ko-", label="integrator")
plt.semilogx(c_fu[0], c_fu[2], "o-", color="gray", label="first-order low-pass")
plt.semilogx(c_fu[0], c_fu[3], "--", color="gray", label=r"$A_\mathrm{oip3}$ (static)")
plt.xlabel(r"$f_\mathrm{u}/f_0$")
plt.ylabel(r"$A_\mathrm{oip3,sim}$ [dBFS]")
plt.grid(which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()